# 01 · Collection

Downloads the City of Boulder's year-end revenue reports, 2018–2024, into `data/raw/`, following Chapters 8 and 9.

- Each file is downloaded once. A file already in `data/raw/` is not requested again.
- 30 seconds between requests. The portal stopped answering after 13 to 17 requests in a row on October 6, 2026 (Chapter 9); this notebook makes about 11. If it stops answering, wait five minutes.
- Run it from a Jupyter started in an activated `boulder-revenue` (or `webdata`) environment.

In [1]:
import csv
import time
from datetime import date
from pathlib import Path

import requests

# Every request says who is asking; put your own address here
HEADERS = {"User-Agent": "WebDataScience/1.0 (hobr8235@colorado.edu)"}

PORTAL = "https://documents.bouldercolorado.gov/WebLink/"
DATA = Path("..") / "data"
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# The certificate bundle from Chapter 9; copy boulder-ca-bundle.pem into data/ first. Never use verify=False
BUNDLE = DATA / "boulder-ca-bundle.pem"
print("Certificate bundle found:", BUNDLE.exists())

Certificate bundle found: True


## Find the December reports in the year folders

The 2018–2020 reports sit in the Revenue Reports folder itself, with known IDs. The 2021–2024 reports are inside year folders, so list each folder, as the folder page does, and read the document IDs from the answer.

In [2]:
def list_folder(folder_id):
    """Return the entries in one portal folder, from the JSON the folder page itself requests."""
    body = {"repoName": "LF8PROD2", "folderId": folder_id, "getNewListing": True,
            "start": 0, "end": 100, "sortColumn": "", "sortAscending": True}
    response = requests.post(PORTAL + "FolderListingService.aspx/GetFolderListing2",
                             json=body, headers=HEADERS, timeout=60, verify=BUNDLE)
    response.raise_for_status()
    return response.json()["data"]["results"]

YEAR_FOLDERS = {2021: 194346, 2022: 194359, 2023: 194372, 2024: 194385}

listings = {}
for i, (year, folder_id) in enumerate(YEAR_FOLDERS.items()):
    if i > 0:
        time.sleep(30)  # wait between requests to the city's server
    listings[year] = list_folder(folder_id)
    print(f"--- {year} folder: {len(listings[year])} entries")
    for entry in listings[year]:
        print(entry["entryId"], entry["name"])

--- 2021 folder: 12 entries
194347 Revenue Report 2021.01
194348 Revenue Report 2021.02
194349 Revenue Report 2021.03
194350 Revenue Report 2021.04
194351 Revenue Report 2021.05
194352 Revenue Report 2021.06
194353 Revenue Report 2021.07
194354 Revenue Report 2021.08
194355 Revenue Report 2021.09
194356 Revenue Report 2021.10
194357 Revenue Report 2021.11
194358 Revenue Report 2021.12
--- 2022 folder: 12 entries
194360 Revenue Report 2022.01
194361 Revenue Report 2022.02
194362 Revenue Report 2022.03
194363 Revenue Report 2022.04
194364 Revenue Report 2022.05
194365 Revenue Report 2022.06
194366 Revenue Report 2022.07
194367 Revenue Report 2022.08
194368 Revenue Report 2022.09
194369 Revenue Report 2022.10
194370 Revenue Report 2022.11
194371 Revenue Report 2022.12
--- 2023 folder: 12 entries
194373 Revenue Report 2023.01
194374 Revenue Report 2023.02
194375 Revenue Report 2023.03
194376 Revenue Report 2023.04
194377 Revenue Report 2023.05
194378 Revenue Report 2023.06
194379 Revenue R

Read the listings above and fill in the December report's ID for each year. Chapter 9 names the December 2024 report `Revenue Report 2024.12`, ID 194397; check that the other years follow the same naming before trusting it.

In [3]:
REPORTS = {
    "revenue-2018.pdf": 194398,
    "revenue-2019.pdf": 194399,
    "revenue-2020.pdf": 194400,
    "revenue-2021-12.pdf": 194358,
    "revenue-2022-12.pdf": 194371,
    "revenue-2023-12.pdf": 194384,
    "revenue-2024-12.pdf": 194397,
}
missing = [name for name, docid in REPORTS.items() if docid is None]
print("Still to fill in:", missing or "none")

Still to fill in: none


## Download

Each download is checked twice: the status code, and the first five bytes, since the portal can answer 200 with an empty body (Chapter 9). Every file downloaded is recorded in `data/raw/download-log.csv`.

In [4]:
log_path = RAW / "download-log.csv"
new_log = not log_path.exists()

with open(log_path, "a", newline="") as log_file:
    log = csv.writer(log_file)
    if new_log:
        log.writerow(["file", "docid", "date", "status", "bytes"])

    first_request = True
    for filename, docid in REPORTS.items():
        path = RAW / filename
        if docid is None:
            print(f"{filename}: no ID yet; skipped")
            continue
        if path.exists():
            print(f"{filename}: already saved")
            continue
        if not first_request:
            time.sleep(30)  # wait between requests to the city's server
        first_request = False

        url = PORTAL + f"ElectronicFile.aspx?docid={docid}&dbid=0&repo=LF8PROD2"
        response = requests.get(url, headers=HEADERS, timeout=120, verify=BUNDLE)
        response.raise_for_status()
        if not response.content.startswith(b"%PDF-"):
            print(f"{filename}: not a PDF ({response.headers.get('Content-Type')}, {len(response.content)} bytes); skipped")
            continue

        path.write_bytes(response.content)
        log.writerow([filename, docid, date.today().isoformat(), response.status_code, len(response.content)])
        print(f"{filename}: {response.status_code}, {len(response.content):,} bytes")

revenue-2018.pdf: already saved
revenue-2019.pdf: already saved
revenue-2020.pdf: already saved
revenue-2021-12.pdf: 200, 913,457 bytes
revenue-2022-12.pdf: 200, 635,976 bytes
revenue-2023-12.pdf: 200, 1,797,079 bytes
revenue-2024-12.pdf: 200, 723,920 bytes
